In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/dataset_all_points.csv')
print(df['class'].unique()) # See all available exercises

['rest' 'right_bicep' 'left_bicep' 'left_shoulder' 'right_shoulder'
 'right_tricep' 'left_tricep']


In [13]:
df_curl = df[df['class'] == 'bicep_curl'].copy()

In [14]:
def calculate_angle(a, b, c):
    a = np.array(a) # First point
    b = np.array(b) # Mid point
    c = np.array(c) # End point

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

# Get landmarks for left elbow angle
# The dataset stores coordinates for each landmark (e.g., x1, y1, z1, v1 for landmark 1)
# MediaPipe landmarks: 11=L_Shoulder, 13=L_Elbow, 15=L_Wrist

# Create empty lists to store the angles
left_elbow_angles = []
for index, row in df_curl.iterrows():
    l_shoulder = [row['x11'], row['y11']]
    l_elbow = [row['x13'], row['y13']]
    l_wrist = [row['x15'], row['y15']]
    angle = calculate_angle(l_shoulder, l_elbow, l_wrist)
    left_elbow_angles.append(angle)

# Add the new feature to the dataframe
df_curl['left_elbow_angle'] = left_elbow_angles

In [15]:
# --- Start replacing from here ---

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Add this line to be 100% sure of the column names
print("Columns in df_curl:", df_curl.columns)

# Features (X) and Target (y)
# The feature is the angle we calculated.
X = df_curl[['left_elbow_angle']] 

# The target is the 'label' column which contains 'correct' or 'incorrect'.
# Let's use the correct column name.
y = df_curl['label'] 

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- End replacing here ---

Columns in df_curl: Index(['class', 'x1', 'y1', 'z1', 'v1', 'x2', 'y2', 'z2', 'v2', 'x3',
       ...
       'v31', 'x32', 'y32', 'z32', 'v32', 'x33', 'y33', 'z33', 'v33',
       'left_elbow_angle'],
      dtype='object', length=134)


KeyError: 'label'

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle

# --------------------------------------------------------------------------
## 1. Data Loading and Cleaning
# --------------------------------------------------------------------------

# Load the dataset
df = pd.read_csv('data/dataset_all_points.csv')

# Create the 'label' column based on whether 'class' contains '_correct'
df['label'] = np.where(df['class'].str.contains('_correct'), 'correct', 'incorrect')

# Create a clean 'exercise' column by removing the suffixes
df['exercise'] = df['class'].str.replace('_correct', '').str.replace('_incorrect', '')

# --- THIS IS THE CORRECTED FILTERING STEP ---
# Filter for both left and right bicep exercises
df_curl = df[df['exercise'].isin(['left_bicep', 'right_bicep'])].copy()

print("Data successfully cleaned and filtered for bicep curls.")
print(f"Found {len(df_curl)} rows for bicep curls.")
print("Unique labels found:", df_curl['label'].unique())

# --------------------------------------------------------------------------
## 2. Feature Engineering (Angle Calculation)
# --------------------------------------------------------------------------

def calculate_angle(a, b, c):
    """Calculates the angle between three 2D points."""
    a = np.array(a) # First point
    b = np.array(b) # Mid point
    c = np.array(c) # End point
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

# Calculate elbow angles for each row in the filtered dataframe
# MediaPipe landmarks: 11=L_Shoulder, 13=L_Elbow, 15=L_Wrist
left_elbow_angles = []
for index, row in df_curl.iterrows():
    l_shoulder = [row['x11'], row['y11']]
    l_elbow = [row['x13'], row['y13']]
    l_wrist = [row['x15'], row['y15']]
    angle = calculate_angle(l_shoulder, l_elbow, l_wrist)
    left_elbow_angles.append(angle)

# Add the new feature to the dataframe
df_curl['left_elbow_angle'] = left_elbow_angles
print("\nSuccessfully calculated and added 'left_elbow_angle' feature.")

# --------------------------------------------------------------------------
## 3. Model Training
# --------------------------------------------------------------------------

# Define features (X) and target (y)
X = df_curl[['left_elbow_angle']]
y = df_curl['label']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize and train the Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel training complete. Accuracy: {accuracy:.2f}")

# --------------------------------------------------------------------------
## 4. Save the Trained Model
# --------------------------------------------------------------------------

with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("\nModel successfully saved to model.pkl")

Data successfully cleaned and filtered for bicep curls.
Found 804 rows for bicep curls.
Unique labels found: ['incorrect']

Successfully calculated and added 'left_elbow_angle' feature.

Model training complete. Accuracy: 1.00

Model successfully saved to model.pkl


In [ ]:
# --- Code to Find a Good Exercise ---

# Reload the original dataframe
df_full = pd.read_csv('data/dataset_all_points.csv')

# Create the label and exercise columns again
df_full['label'] = np.where(df_full['class'].str.contains('_correct'), 'correct', 'incorrect')
df_full['exercise'] = df_full['class'].str.replace('_correct', '').str.replace('_incorrect', '')

# Group by exercise and check the labels present for each
exercise_labels = df_full.groupby('exercise')['label'].unique().apply(list)

print("--- Exercise Label Analysis ---")
for exercise, labels in exercise_labels.items():
    if 'correct' in labels and 'incorrect' in labels:
        print(f"✅ {exercise}: Has BOTH correct and incorrect labels.")
    else:
        print(f"❌ {exercise}: Has ONLY {labels} labels.")

--- Exercise Label Analysis ---
❌ left_bicep: Has ONLY ['incorrect'] labels.
❌ left_shoulder: Has ONLY ['incorrect'] labels.
❌ left_tricep: Has ONLY ['incorrect'] labels.
❌ rest: Has ONLY ['incorrect'] labels.
❌ right_bicep: Has ONLY ['incorrect'] labels.
❌ right_shoulder: Has ONLY ['incorrect'] labels.
❌ right_tricep: Has ONLY ['incorrect'] labels.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle

# --------------------------------------------------------------------------
## 1. Data Loading
# --------------------------------------------------------------------------

# Load the dataset
df = pd.read_csv('data/dataset_all_points.csv')
print("Dataset loaded successfully.")
print("Available exercise classes:", df['class'].unique())

# --------------------------------------------------------------------------
## 2. Feature and Target Preparation
# --------------------------------------------------------------------------

# The target (y) is the 'class' column itself, which contains the exercise names.
y = df['class']

# The features (X) will be all the x, y, and z coordinates for the 33 landmarks.
# We will exclude the 'class' column and the visibility ('v') columns.
feature_columns = [col for col in df.columns if col != 'class' and not col.startswith('v')]
X = df[feature_columns]

print(f"\nTraining with {len(feature_columns)} features.")
print(f"Target classes: {y.nunique()}")

# --------------------------------------------------------------------------
## 3. Model Training for Exercise Classification
# --------------------------------------------------------------------------

# Split data into training and testing sets.
# 'stratify=y' ensures both train and test sets have a proportional
# representation of each exercise class, which is crucial for good performance.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize and train the Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel training complete. Accuracy: {accuracy:.2f}")

# Print a detailed report showing performance for each exercise class
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# --------------------------------------------------------------------------
## 4. Save the Trained Model
# --------------------------------------------------------------------------

with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("\nClassifier model successfully saved to model.pkl")

# --------------------------------------------------------------------------
## 5. Calculate and Save Prototype Poses for Q-Score
# --------------------------------------------------------------------------

print("\nCalculating prototype poses for consistency scoring...")

# Calculate the mean of all feature columns for each exercise class
prototype_poses = df.groupby('class')[feature_columns].mean()

# Save the prototypes to a CSV file
prototype_poses.to_csv('prototype_poses.csv')

print("\nPrototype poses successfully saved to prototype_poses.csv")
print("\n--- Training Complete ---")
print("You now have 'model.pkl' and 'prototype_poses.csv' ready for the Streamlit app.")

Dataset loaded successfully.
Available exercise classes: ['rest' 'right_bicep' 'left_bicep' 'left_shoulder' 'right_shoulder'
 'right_tricep' 'left_tricep']

Training with 99 features.
Target classes: 7

Model training complete. Accuracy: 1.00

Classification Report:
                precision    recall  f1-score   support

    left_bicep       1.00      1.00      1.00        87
 left_shoulder       1.00      1.00      1.00        75
   left_tricep       1.00      1.00      1.00        63
          rest       1.00      1.00      1.00        81
   right_bicep       1.00      1.00      1.00        74
right_shoulder       1.00      1.00      1.00        80
  right_tricep       1.00      1.00      1.00        80

      accuracy                           1.00       540
     macro avg       1.00      1.00      1.00       540
  weighted avg       1.00      1.00      1.00       540


Classifier model successfully saved to model.pkl

Calculating prototype poses for consistency scoring...

Prototy

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle

# --------------------------------------------------------------------------
## 1. Data Loading
# --------------------------------------------------------------------------

# Load the dataset
df = pd.read_csv('data/dataset_all_points.csv')
print("Dataset loaded successfully.")
print("Available exercise classes:", df['class'].unique())

# --------------------------------------------------------------------------
## 2. Feature and Target Preparation
# --------------------------------------------------------------------------

# The target (y) is the 'class' column, which contains the exercise names.
y = df['class']

# The features (X) will be all the x, y, and z coordinates for the 33 landmarks.
# We will exclude the 'class' column and the visibility ('v') columns.
feature_columns = [col for col in df.columns if col != 'class' and not col.startswith('v')]
X = df[feature_columns]

print(f"\nTraining with {len(feature_columns)} features.")
print(f"Target classes: {y.nunique()}")

# --------------------------------------------------------------------------
## 3. Model Training for Exercise Classification
# --------------------------------------------------------------------------

# Split data into training and testing sets.
# 'stratify=y' ensures both train and test sets have a proportional
# representation of each exercise class.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.7, random_state=42, stratify=y)

# Initialize and train the Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel training complete. Accuracy: {accuracy:.2f}")

# Print a detailed report showing performance for each exercise class
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# --------------------------------------------------------------------------
## 4. Save the Trained Model
# --------------------------------------------------------------------------

with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("\nClassifier model successfully saved to model.pkl")

# --------------------------------------------------------------------------
## 5. Calculate and Save NORMALIZED Prototype Poses for Q-Score
# --------------------------------------------------------------------------

print("\nCalculating NORMALIZED prototype poses for consistency scoring...")

# Create a copy to work with
df_normalized = df.copy()

# --- Normalization Logic ---
# This makes the pose data independent of the person's size or position on screen.

# Get coordinates for key landmarks as NumPy arrays
left_hip = df_normalized[['x23', 'y23', 'z23']].values
right_hip = df_normalized[['x24', 'y24', 'z24']].values
left_shoulder = df_normalized[['x11', 'y11', 'z11']].values
right_shoulder = df_normalized[['x12', 'y12', 'z12']].values

# A. Calculate the center of the hips to act as the new origin (0,0,0) for each pose
hip_center = (left_hip + right_hip) / 2

# B. Calculate the shoulder width to act as a scaling factor
# We use np.linalg.norm to get the Euclidean distance between the two shoulder points.
shoulder_width = np.linalg.norm(left_shoulder - right_shoulder, axis=1)

# Avoid division by zero if shoulder width is 0 for any row
shoulder_width[shoulder_width == 0] = 1 

# Normalize all 99 feature columns (33 landmarks * 3 coordinates)
for i in range(1, 34):
    for coord_idx, coord_name in enumerate(['x', 'y', 'z']):
        col = f'{coord_name}{i}'
        # 1. Center the coordinate around the hip center's corresponding axis
        # 2. Scale by the calculated shoulder width
        df_normalized[col] = (df_normalized[col] - hip_center[:, coord_idx]) / shoulder_width

# Now, calculate the mean of these NORMALIZED features for each class
prototype_poses = df_normalized.groupby('class')[feature_columns].mean()

# Save the normalized prototypes to a new CSV file
prototype_poses.to_csv('prototype_poses_normalized.csv')

print("\nNormalized prototype poses saved to 'prototype_poses_normalized.csv'")

print("\n--- Training Complete ---")
print("You now have 'model.pkl' and 'prototype_poses_normalized.csv' ready for the Streamlit app.")

Dataset loaded successfully.
Available exercise classes: ['rest' 'right_bicep' 'left_bicep' 'left_shoulder' 'right_shoulder'
 'right_tricep' 'left_tricep']

Training with 99 features.
Target classes: 7

Model training complete. Accuracy: 1.00

Classification Report:
                precision    recall  f1-score   support

    left_bicep       1.00      1.00      1.00       305
 left_shoulder       1.00      1.00      1.00       261
   left_tricep       1.00      1.00      1.00       222
          rest       1.00      1.00      1.00       284
   right_bicep       1.00      1.00      1.00       258
right_shoulder       1.00      1.00      1.00       281
  right_tricep       1.00      1.00      1.00       279

      accuracy                           1.00      1890
     macro avg       1.00      1.00      1.00      1890
  weighted avg       1.00      1.00      1.00      1890


Classifier model successfully saved to model.pkl

Calculating NORMALIZED prototype poses for consistency scoring.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle

# --------------------------------------------------------------------------
## 1. Data Loading
# --------------------------------------------------------------------------

# Load the dataset
df = pd.read_csv('data/dataset_all_points.csv')
print("Dataset loaded successfully.")
print("Available exercise classes:", df['class'].unique())

# --------------------------------------------------------------------------
## 2. Feature and Target Preparation
# --------------------------------------------------------------------------

# The target (y) is the 'class' column, which contains the exercise names.
y = df['class']

# The features (X) will be all the x, y, and z coordinates for the 33 landmarks.
# We will exclude the 'class' column and the visibility ('v') columns.
feature_columns = [col for col in df.columns if col != 'class' and not col.startswith('v')]
X = df[feature_columns]

print(f"\nTraining with {len(feature_columns)} features.")
print(f"Target classes: {y.nunique()}")

# --------------------------------------------------------------------------
## 3. Model Training for Exercise Classification (with Regularization)
# --------------------------------------------------------------------------

# Split data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- THIS IS THE MODIFIED LINE ---
# We add 'max_depth=5' to prevent overfitting and make the model more general.
# This will intentionally reduce the accuracy from 1.00 to a more realistic number.
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=3)
# ---------------------------------

model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel training complete. Accuracy: {accuracy:.2f}")

# Print a detailed report showing performance for each exercise class
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# --------------------------------------------------------------------------
## 4. Save the Trained Model
# --------------------------------------------------------------------------

with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("\nClassifier model successfully saved to model.pkl")

# --------------------------------------------------------------------------
## 5. Calculate and Save NORMALIZED Prototype Poses for Q-Score
# --------------------------------------------------------------------------

print("\nCalculating NORMALIZED prototype poses for consistency scoring...")

# Create a copy to work with
df_normalized = df.copy()

# Get coordinates for key landmarks as NumPy arrays
left_hip = df_normalized[['x23', 'y23', 'z23']].values
right_hip = df_normalized[['x24', 'y24', 'z24']].values
left_shoulder = df_normalized[['x11', 'y11', 'z11']].values
right_shoulder = df_normalized[['x12', 'y12', 'z12']].values

# Calculate the center of the hips to act as the new origin for each pose
hip_center = (left_hip + right_hip) / 2

# Calculate the shoulder width to act as a scaling factor
shoulder_width = np.linalg.norm(left_shoulder - right_shoulder, axis=1)
shoulder_width[shoulder_width == 0] = 1 # Avoid division by zero

# Normalize all 99 feature columns
for i in range(1, 34):
    for coord_idx, coord_name in enumerate(['x', 'y', 'z']):
        col = f'{coord_name}{i}'
        df_normalized[col] = (df_normalized[col] - hip_center[:, coord_idx]) / shoulder_width

# Calculate the mean of these NORMALIZED features for each class
prototype_poses = df_normalized.groupby('class')[feature_columns].mean()

# Save the normalized prototypes to a new CSV file
prototype_poses.to_csv('prototype_poses_normalized.csv')
print("\nNormalized prototype poses saved to 'prototype_poses_normalized.csv'")

# Calculate and save the average shoulder width
print("\nCalculating average shoulder width...")
avg_shoulder_width = np.mean(shoulder_width)

with open('avg_shoulder_width.txt', 'w') as f:
    f.write(str(avg_shoulder_width))
print("Average shoulder width saved to 'avg_shoulder_width.txt'")

print("\n--- Training Complete ---")
print("You now have 'model.pkl', 'prototype_poses_normalized.csv', and 'avg_shoulder_width.txt' ready.")

Dataset loaded successfully.
Available exercise classes: ['rest' 'right_bicep' 'left_bicep' 'left_shoulder' 'right_shoulder'
 'right_tricep' 'left_tricep']

Training with 99 features.
Target classes: 7

Model training complete. Accuracy: 1.00

Classification Report:
                precision    recall  f1-score   support

    left_bicep       1.00      1.00      1.00        87
 left_shoulder       1.00      1.00      1.00        75
   left_tricep       1.00      1.00      1.00        63
          rest       1.00      1.00      1.00        81
   right_bicep       1.00      1.00      1.00        74
right_shoulder       1.00      1.00      1.00        80
  right_tricep       1.00      1.00      1.00        80

      accuracy                           1.00       540
     macro avg       1.00      1.00      1.00       540
  weighted avg       1.00      1.00      1.00       540


Classifier model successfully saved to model.pkl

Calculating NORMALIZED prototype poses for consistency scoring.

In [ ]:
from sklearn.metrics import classification_report

print("\n--- Table 1: Classification Performance Report ---")
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=model.classes_))


--- Table 1: Classification Performance Report ---


NameError: name 'model' is not defined